In [ ]:
from web3 import Web3
import json
import os

infura_key = '2e306bdddc7843108fe30334b2dfcfb2'
wallet_public_address = ''
wallet_private_key = ''

USDC_address = Web3.to_checksum_address('0x94a9D9AC8a22534E3FaCa9F4e7F2E2cf85d5E4C8')
USDT_address = Web3.to_checksum_address('0xaA8E23Fb1079EA71e0a56F48a2aA51851D8433D0') 
USTUSD_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')

wallet_public_address = Web3.to_checksum_address(wallet_public_address)

# Change this to use your own RPC URL for Sepolia Testnet
web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))

print("Connected to Sepolia Testnet:", web3)

abi_file_path = os.path.join('./abis.json')
try:
    with open(abi_file_path, 'r', encoding='utf-8') as f:
        abi_data = json.load(f)
    print("ABI loaded successfully.")
except Exception as e:
    print(f"Error loading ABI: {e}")

Connected to Sepolia Testnet: <web3.main.Web3 object at 0x000001675EFB02B0>
ABI loaded successfully.


## Check the current price of USDT in USTUSD from Uniswap V3 Pool.

In [16]:
factory_addr = '0x0227628f3F023bb0B980b67D528571c95c6DaC1c'
factory_contract = web3.eth.contract(factory_addr, abi=abi_data['UNISWAP_FACTORY_ABI'])
def get_pool_address(tokenA, tokenB, tier_fee, factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    # tier_fee: 100 for 0.01%, 500 for 0.05%, 3000 for 0.3%, 10000 for 1%
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, tier_fee).call()
    return pool_address

USDT_USTUSD_pool_address = get_pool_address(USTUSD_address, USDT_address, 500, factory_contract)
print("USDT-USTUSD Pool Address:", USDT_USTUSD_pool_address)

USDT_USTUSD_pool = web3.eth.contract(address=USDT_USTUSD_pool_address, abi=abi_data['UNISWAP_V3_POOL_ABI'])

USDT_price_in_USTUSD = USDT_USTUSD_pool.functions.slot0().call()[0]**2 / 2**192 / (10**12)
print(f"1 USDT = {USDT_price_in_USTUSD:.6f} USTUSD")

USDT-USTUSD Pool Address: 0x49e75DCCCf6Bb59531dC52Ea85579Bc460A59ccF
1 USDT = 8.028020 USTUSD


## Perform the swap on Uniswap V3 Pool

- original number of USDT in the pool: 51517.079575
- original number of USTUSD in the pool: 412100.796150338963559755

In [13]:
universal_router_address = web3.to_checksum_address('0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD')
universal_router_contract = web3.eth.contract(address=universal_router_address, abi=abi_data['UNIVERSAL_ROUTER_ABI'])
permit2_address = web3.to_checksum_address('0x000000000022D473030F116dDEE9F6B43aC78BA3')
permit2_contract = web3.eth.contract(address=permit2_address, abi=abi_data['PERMIT2_ABI'])
USDT_contract = web3.eth.contract(address=USDT_address, abi=abi_data['ERC20_ABI'])

tx = USDT_contract.functions.approve(permit2_address, 2**256 - 1).build_transaction({
    "from": wallet_public_address,
    "nonce": web3.eth.get_transaction_count(wallet_public_address),
})

signed_txn = web3.eth.account.sign_transaction(tx, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
tx_hash = web3.eth.send_raw_transaction(raw_transaction)
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
print(f"Approval transaction hash: {tx_hash.hex()}")

if receipt.status == 1:
    print("Approval executed successfully!")
else:
    print("Approval failed.")

tx = permit2_contract.functions.approve(USDT_address, universal_router_address, 2**160 - 1, 2**48 - 1).build_transaction({
    "from": wallet_public_address,
    "nonce": web3.eth.get_transaction_count(wallet_public_address),
})
signed_txn = web3.eth.account.sign_transaction(tx, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
tx_hash = web3.eth.send_raw_transaction(raw_transaction)
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
print(f"Permit2 approval transaction hash: {tx_hash.hex()}")

if receipt.status == 1:
    print("Permit2 approval executed successfully!")
else:
    print("Permit2 approval failed.")


Approval transaction hash: d17739148d3648a207f85fa57020267cc40929a9486db9850970c37d899803af
Approval executed successfully!
Permit2 approval transaction hash: 44a01cd2cb94a48c47ae248534ca573dcaaa3b6f02535e01e7868977d9942f6f
Permit2 approval executed successfully!


In [ ]:
from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
codec = RouterCodec()

USDT_in_amount = 20000 * 10**6  # 

encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        USDT_in_amount,  # amount in, (20000 USDT with 6 decimals)
        0,
        [   
            USDT_address,
            500,
            USTUSD_address
        ],
    ).build(2**256 - 1)

trx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

signed_txn = web3.eth.account.sign_transaction(trx_params, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
txn_hash = web3.eth.send_raw_transaction(raw_transaction)
receipt = web3.eth.wait_for_transaction_receipt(txn_hash)
print("Hash of universal router swap transaction : ", web3.to_hex(txn_hash))

if receipt.status == 1:
    print("Swap executed successfully!")
else:
    print("Swap failed.")

Hash of universal router swap transaction :  0xafc64b001ce459d21e0f6c4a4dd927efde9663b6d4552f7ead8bde92b33baa56


## Price back

In [14]:
USTUSD_contract = web3.eth.contract(address=USTUSD_address, abi=abi_data['ERC20_ABI'])

tx = USTUSD_contract.functions.approve(permit2_address, 2**256 - 1).build_transaction({
    "from": wallet_public_address,
    "nonce": web3.eth.get_transaction_count(wallet_public_address),
})

signed_txn = web3.eth.account.sign_transaction(tx, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
tx_hash = web3.eth.send_raw_transaction(raw_transaction)
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
print(f"Approval transaction hash: {tx_hash.hex()}")
if receipt.status == 1:
    print("Approval executed successfully!")

tx = permit2_contract.functions.approve(USTUSD_address, universal_router_address, 2**160 - 1, 2**48 - 1).build_transaction({
    "from": wallet_public_address,
    "nonce": web3.eth.get_transaction_count(wallet_public_address),
})
signed_txn = web3.eth.account.sign_transaction(tx, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
tx_hash = web3.eth.send_raw_transaction(raw_transaction)
receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
print(f"Permit2 approval transaction hash: {tx_hash.hex()}")
if receipt.status == 1:
    print("Permit2 approval executed successfully!")

Approval transaction hash: 2d14f7c06512be6e13bf8e896a4eb8b85a12b5762038d239d634104a72445074
Approval executed successfully!
Permit2 approval transaction hash: c890f8693c39c6c8936816381ecf4deae025b9d03152d94f6e211474bcc1117c
Permit2 approval executed successfully!


In [15]:
USTUSD_in_amount = int(116000.907363250264747423 * 10**18)

encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        USTUSD_in_amount,  # amount in, (116000.907363250264747423 USTUSD with 18 decimals)
        0,
        [   
            USTUSD_address,
            500,
            USDT_address
        ],
    ).build(2**256 - 1)

trx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

signed_txn = web3.eth.account.sign_transaction(trx_params, wallet_private_key)
raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
txn_hash = web3.eth.send_raw_transaction(raw_transaction)
receipt = web3.eth.wait_for_transaction_receipt(txn_hash)
print("Hash of universal router swap transaction : ", web3.to_hex(txn_hash))

if receipt.status == 1:
    print("Swap executed successfully!")
else:
    print("Swap failed.")


Hash of universal router swap transaction :  0x1e6f6c1e17ac97a4b9ebbd3e08dc80527152c98753ddb9b5865a40e3f5f2be62
Swap executed successfully!
